In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import shutil
import json
from IPython.display import display

## Объединение классов в более крупные группы

In [ ]:
# Объединение классов в более крупные группы
CLASS_RULES = {
    # УДАЛЯЕМ
    'Алевролит_песчанистый': None,
    'Песчаник_с_включениями_угля': None,
    'Известняк': None,
    'Породы_фундамента': None,


    # ГРУППА: Аргиллит
    'Аргиллит': 'Аргиллит', 
    
    
    # ГРУППА: Переслаивание_песчаника,_аргиллита_и_алевролита
    'Переслаивание_песчаника,_аргиллита_и_алевролита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Песчаник_с_включениями_алевролита_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Песчаник_с_включениями_алевролита_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Чередование_аргиллита,_алевролита_и_песчаника': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Алевролит_с_прослоями_песчаника_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Песчаник_с_прослоями_алевролита_и_аргиллита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Аргиллит_с_прослоями_песчаника_и_алевролита': 'Переслаивание_песчаника,_аргиллита_и_алевролита',

    # ГРУППА: Уголь,_уголь_с_прослоями_аргиллита
    'Аргиллит_углистый': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Уголь_с_прослоями_аргиллита': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Уголь': 'Уголь,_уголь_с_прослоями_аргиллита',
    'Аргиллит_с_включениями_угля': 'Уголь,_уголь_с_прослоями_аргиллита',

    # ГРУППА: Глинисто-карбонатная_порода
    'Опока_глинистая': 'Глинисто-карбонатная_порода',
    'Глинисто-карбонатная_порода': 'Глинисто-карбонатная_порода',
    'Глина_опоковидная': 'Глинисто-карбонатная_порода',
    'Глина_аргиллитоподобная_с_прослоями_глины_опоковидной': 'Глинисто-карбонатная_порода',
    'Кремнисто-глинистая_порода': 'Глинисто-карбонатная_порода',
    'Глина_опоковидная_с_включением_глинистых_опок': 'Глинисто-карбонатная_порода',
    'Глина_аргиллитоподобная': 'Глинисто-карбонатная_порода',

    # ГРУППА: Алевролит
    'Алевролит': 'Алевролит',
    'Алевролит_с_включениями_угля': 'Алевролит',
    'Алевролит_глинистый': 'Алевролит',
    'Алевролит_карбонатный': 'Алевролит',

    # ГРУППА: Переслаивание_аргиллита_и_алевролита
    'Переслаивание_аргиллита_и_алевролита': 'Переслаивание_аргиллита_и_алевролита',
    'Аргиллит_алевритовый': 'Переслаивание_аргиллита_и_алевролита',
    'Алевролит_с_прослоями_аргиллита': 'Переслаивание_аргиллита_и_алевролита',
    'Аргиллит_с_прослоями_алевролита': 'Переслаивание_аргиллита_и_алевролита',

    # ГРУППА: Песчаник_с_прослоями_алевролита
    'Песчаник_с_прослоями_алевролита': 'Песчаник_с_прослоями_алевролита',
    'Переслаивание_песчаника_и_алевролита': 'Песчаник_с_прослоями_алевролита',
    'Алевролит_с_прослоями_песчаника': 'Песчаник_с_прослоями_алевролита',

    # ГРУППА: Песчаник_с_прослоями_аргиллита
    'Песчаник_с_прослоями_аргиллита': 'Песчаник_с_прослоями_аргиллита',
    'Аргиллит_с_прослоями_песчаника': 'Песчаник_с_прослоями_аргиллита',
    'Переслаивание_песчаника_и_аргиллита': 'Песчаник_с_прослоями_аргиллита',

    # ГРУППА: Песчаник
    'Песчаник': 'Песчаник',
    'Песчаник_карбонатный': 'Песчаник',
}

# Итого по классам
CLASSES_ORDER = [
    'Песчаник',
    'Аргиллит', 
    'Алевролит',
    'Переслаивание_песчаника,_аргиллита_и_алевролита',
    'Переслаивание_аргиллита_и_алевролита',
    'Песчаник_с_прослоями_алевролита',
    'Песчаник_с_прослоями_аргиллита',
    'Глинисто-карбонатная_порода',
    'Уголь,_уголь_с_прослоями_аргиллита'
]






## Пути к данным

In [ ]:
r = Path.cwd().parent.parent
ROOT = r / Path(r"data\Digital_core_v4")
TARGET_ROOT = r / Path(r"data\Digital_core_tmv2")
output_path = TARGET_ROOT / "label_encoder.json"
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


## Разделение на train, val, test

In [ ]:

VAL_WELLS = {"Харасавэйск_1400", "Харасавэйск_1900", "Харасавэйск_2000"}
TRAIN_WELLS = {
    "Восточно-Падинск_3-ВП", "Соболох-Неджелинск_3", "Таб-Яхинская_10360",
    "Харасавэйск_300", "Харасавэйск_600 к 6", "Харасавэйск_700",
    "Харасавэйск_900", "Харасавэйск_1100", "Харасавэйск_1800",
    "Ю-Песцовый лу_12", "Ямбург_1 N", "Ямбургск_24604"
}

TEST_WELLS = {"Харасавэйск_1700"}

ALL_WELLS = TRAIN_WELLS | VAL_WELLS | TEST_WELLS



## Полезные функции

In [ ]:
def extract_class(filename):
    """Извлекает название класса из имени файла (до первой цифры)"""
    stem = Path(filename).stem
    parts = stem.split('_')
    for i, part in enumerate(parts):
        if part and part[0].isdigit():
            return '_'.join(parts[:i])
    return None

def find_well(dirpath):
    """Ищет название скважины в пути"""
    for part in Path(dirpath).parts:
        if part in ALL_WELLS:
            return part
    return None

def get_split(well):
    """Определяет сплит по скважине: train/val/test"""
    if well in TRAIN_WELLS: return 'train'
    if well in VAL_WELLS: return 'val'
    if well in TEST_WELLS: return 'test'
    return None

def get_group_total(subclasses):
    """Подсчёт общего количества образцов в группе"""
    return sum(d["count"] for d in subclasses.values())

def safe_image_open(path):
    """Безопасное открытие изображения с обработкой ошибок"""
    try:
        img = Image.open(path)
        img.load()
        return img
    except Exception:
        return None

In [ ]:
# Сканирование модальности ДС
hierarchy = defaultdict(lambda: defaultdict(lambda: {"count": 0, "all_samples": []}))
total_processed = 0
total_deleted = 0

print("Сканирую ТОЛЬКО модальность ДС и применяю правила...")

for dirpath, _, filenames in os.walk(ROOT):
    if 'ДС' not in Path(dirpath).parts:
        continue
        
    for f in filenames:
        if Path(f).suffix.lower() not in IMG_EXTS:
            continue
            
        #  Извлекаем класс через функцию из конфига
        original_cls = extract_class(f)
        if not original_cls:
            continue
            
        #  Предупреждение для неизвестных классов
        if original_cls not in CLASS_RULES:
            print(f"  Неизвестный класс '{original_cls}' в файле {f}")
        
        # Применяем правила группировки
        target = CLASS_RULES.get(original_cls, original_cls)
        if target is None:
            total_deleted += 1
            continue
            
        final_group = target
        data = hierarchy[final_group][original_cls]
        data["count"] += 1
        data["all_samples"].append(Path(dirpath) / f)
        total_processed += 1
# === Сортировка и вывод статистики ===
sorted_groups = sorted(hierarchy.items(), key=lambda x: get_group_total(x[1]), reverse=True)
print(f'\nСканирование ДС завершено.')
print(f'   Обработано файлов: {total_processed} | Удалено: {total_deleted}')
print(f'   Осталось групп: {len(sorted_groups)}\n')


## Визуализация классов

In [ ]:
# === ВИЗУАЛИЗАЦИЯ (случайные 5 фото на подкласс, с фиксированным seed) ===
rng = random.Random(RANDOM_SEED)

rows = []
for group_name, subclasses in sorted_groups:
    total = get_group_total(subclasses)
    rows.append(("GROUP_HEADER", group_name, total, None))
    
    sorted_subs = sorted(subclasses.items(), key=lambda x: x[1]["count"], reverse=True)
    for sub_name, data in sorted_subs:
        all_samples = data["all_samples"]
        # Берём 5 случайных, но всегда одни и те же при перезапуске
        display_samples = rng.sample(all_samples, min(5, len(all_samples)))
        rows.append(("SUBCLASS", sub_name, data["count"], display_samples))

# Отрисовка
fig_height = sum(3.2 if row[0] == "GROUP_HEADER" else 2.0 for row in rows) + 1
fig, axes = plt.subplots(len(rows), 6, figsize=(28, fig_height))
if len(rows) == 1:
    axes = [axes]

row_idx = 0
for row_type, name, count, samples in rows:
    if row_type == "GROUP_HEADER":
        for col in range(6):
            axes[row_idx, col].axis('off')
            if col == 0:
                axes[row_idx, col].text(0, 0.5, f"[{name}]\nВсего: {count}", 
                                       fontsize=11, fontweight='bold', va='center')
                axes[row_idx, col].add_patch(plt.Rectangle((0,0), 1, 1, facecolor='lightblue', alpha=0.15, transform=axes[row_idx, col].transAxes))
    else:
        axes[row_idx, 0].axis('off')
        axes[row_idx, 0].text(0, 0.5, f"{name}\n({count})", fontsize=9, va='center')
        
        for col in range(5):
            ax = axes[row_idx, col+1]
            ax.set_aspect('equal')
            if col < len(samples):
                img = safe_image_open(samples[col])
                if img:
                    ax.imshow(img)
                else:
                    ax.text(0.5, 0.5, 'Err', ha='center', va='center', fontsize=8, color='red')
            ax.axis('off')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.margins(0)
    row_idx += 1

plt.tight_layout(pad=0.8, h_pad=1.0, w_pad=0.8)
plt.show()

# === ТЕКСТОВЫЙ ОТЧЕТ ===
print("\n" + "="*85)
print("ИТОГОВАЯ СТРУКТУРА ДАТАСЕТА (только модальность ДС)")
print("="*85)
for group_name, subclasses in sorted_groups:
    total = get_group_total(subclasses)
    print(f'\n{group_name} — {total} изображений')
    sorted_subs = sorted(subclasses.items(), key=lambda x: x[1]["count"], reverse=True)
    for sub_name, data in sorted_subs:
        print(f'   ├─ {sub_name}: {data["count"]}')

final_total = sum(get_group_total(subs) for subs in hierarchy.values())
print(f'\nОбщий объем датасета для обучения (ДС): {final_total} изображений')
print("="*85)

In [ ]:
#  Анализ распределения классов по сплитам — используем функции из Ячейки 0
train_counts = Counter()
val_counts = Counter()
test_counts = Counter()  #  добавили тест

print(" Сканирую модальность ДС для анализа баланса...")

for dirpath, _, filenames in os.walk(ROOT):
    if 'ДС' not in Path(dirpath).parts: 
        continue
    
    well = find_well(dirpath)
    if not well: 
        continue
    
    split = get_split(well)
    if not split:
        continue
    
    for f in filenames:
        if Path(f).suffix.lower() not in IMG_EXTS: 
            continue
            
        orig_cls = extract_class(f)
        if not orig_cls: 
            continue
        
        final_cls = CLASS_RULES.get(orig_cls, orig_cls)
        if final_cls is None: 
            continue
        
        if split == 'train':
            train_counts[final_cls] += 1
        elif split == 'val':
            val_counts[final_cls] += 1
        elif split == 'test':
            test_counts[final_cls] += 1

# === ВЫВОД: таблица баланса ===
print("\n" + "="*90)
print("БАЛАНС КЛАССОВ: TRAIN / VAL / TEST (модальность ДС)")
print("="*90)

print(f"\n{'Класс':<50} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-"*90)

for cls in CLASSES_ORDER:
    t = train_counts.get(cls, 0)
    v = val_counts.get(cls, 0)
    te = test_counts.get(cls, 0)
    print(f"{cls:<50} {t:>8,} {v:>8,} {te:>8,}")

#  Проверка полноты тестовой выборки
missing_in_test = set(CLASSES_ORDER) - set(test_counts.keys())
if missing_in_test:
    print(f"\n В тесте отсутствуют классы: {missing_in_test}")
else:
    print(f"\n Все {len(CLASSES_ORDER)} классов присутствуют в тестовой выборке")

In [ ]:
# === МЕТРИКИ ДИСБАЛАНСА ===
print("\n" + "="*90)
print("  МЕТРИКИ ДИСБАЛАНСА")
print("="*90)

for split_name, counts in [("TRAIN", train_counts), ("VAL", val_counts), ("TEST", test_counts)]:
    if not counts:
        continue
    vals = list(counts.values())
    mx, mn = max(vals), min(vals)
    ratio = mx / mn if mn > 0 else float('inf')
    print(f"\n {split_name}:")
    print(f"   • Максимум: {mx:,} | Минимум: {mn:,}")
    print(f"   • Дисбаланс (макс:мин): {ratio:.1f}:1")
    print(f"   • Среднее: {sum(vals)/len(vals):,.0f} | Медиана: {sorted(vals)[len(vals)//2]:,}")

#  Рекомендации
if train_counts:
    smallest = min(train_counts, key=train_counts.get)
    if train_counts[smallest] < 100:
        print(f"\n  Класс '{smallest}' мал ({train_counts[smallest]} фото)")
        print("   → Рекомендуется: WeightedRandomSampler + class_weights в Loss")

print("\n" + "="*90)

In [ ]:

# ==========================================
# ОСНОВНОЙ ПРОЦЕСС
# ==========================================
stats = defaultdict(int)
total_copied = 0
total_deleted = 0

print(" Сканирую исходную папку и раскладываю файлы...")
for dirpath, _, filenames in os.walk(ROOT):
    # 1. Определяем модальность
    modality = None
    for part in Path(dirpath).parts:
        if part == "ДС": modality = "ДС"; break
        if part == "УФ": modality = "УФ"; break
    if not modality: continue

    # 2. Определяем скважину и сплит
    well = find_well(dirpath)
    if not well: continue
    
    if well in TRAIN_WELLS:
        split = "train"
    elif well in VAL_WELLS:
        split = "val"
    elif well in TEST_WELLS:
        split = "test"
    else:
        continue

    # 3. Копируем файлы
    for f in filenames:
        if Path(f).suffix.lower() not in img_exts: continue
        
        src_file = Path(dirpath) / f
        orig_cls = extract_class(f)
        
        if not orig_cls: continue
            
        # Применяем правила группировки
        target_cls = CLASS_RULES.get(orig_cls, orig_cls)
        
        if target_cls is None:
            total_deleted += 1
            continue

        dest_dir = TARGET_ROOT / modality / split / target_cls
        dest_dir.mkdir(parents=True, exist_ok=True)
        
        shutil.copy2(src_file, dest_dir / f)
        
        stats[f"{modality}_{split}_{target_cls}"] += 1
        total_copied += 1
        
        # [СТИЛЬ]: magic number 2000 — вынеси в константу LOG_EVERY = 2000
        if total_copied % 2000 == 0:
            print(f"  ⏳ Скопировано: {total_copied:,} файлов...", end='\r')

print("\n Готово!")
print(f" Цель: {TARGET_ROOT}")

# 📊 Статистика по итогу
print("\n ИТОГОВАЯ СТАТИСТИКА:")
for mod in ["ДС", "УФ"]:
    for s in ["train", "val", "test"]:
        count = sum(v for k, v in stats.items() if k.startswith(f"{mod}_{s}"))
        print(f"   {mod} / {s}: {count:,} файлов")

print(f"\n Всего удалено (по правилам): {total_deleted}")

In [ ]:
# Создаём маппинг: класс → индекс
class_to_idx = {cls: idx for idx, cls in enumerate(CLASSES_ORDER)}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(class_to_idx, f, ensure_ascii=False, indent=2)

print(f"label_encoder.json сохранён: {output_path}")
print("\nСодержимое файла:")
for cls, idx in class_to_idx.items():
    print(f"   {idx}: {cls}")